# **Environment setup**

In [1]:
!pip install -q "av<14.0.0" faster-whisper sentence-transformers yt-dlp
!apt-get -y install ffmpeg -qq

import os
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment ready. Hardware accelerator: {device}")

Environment ready. Hardware accelerator: cuda


# **Ingesting Sample Video & Extracting Audio**

In [2]:
import subprocess
import yt_dlp


VIDEO_URL = "https://youtu.be/EJPI04bjcOY?si=tajYdSAjfmxbQsAd"

os.makedirs("week1_demo", exist_ok=True)
video_path = "week1_demo/input_video.mp4"
audio_path = "week1_demo/input_audio.wav"


if not os.path.exists(video_path):
    print("Downloading sample video...")
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        'outtmpl': video_path,
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([VIDEO_URL])


subprocess.run([
    "ffmpeg", "-y", "-i", video_path,
    "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
    audio_path
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Audio stream successfully isolated:", audio_path)

Audio stream successfully isolated: week1_demo/input_audio.wav


# **Transcription & Semantic Target Detection (Whisper + Sentence-Transformers)**

In [4]:
from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer, util


print("Loading Whisper...")
compute_type = "float16" if device == "cuda" else "int8"
whisper = WhisperModel("base", device='cpu', compute_type='int8')

segments, _ = whisper.transcribe(audio_path)
transcript = []
for seg in segments:
    transcript.append({
        "start": round(seg.start, 2),
        "end": round(seg.end, 2),
        "text": seg.text.strip()
    })

print(f"Extracted {len(transcript)} timestamped speech segments.")


embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

target_categories = {
    "Product Unboxing": "opening packaging unboxing accessories in the box retail seal peel",
    "Feature Demonstration": "camera sample test gaming performance screen display benchmark testing",
    "Final Verdict": "conclusion final thoughts should you buy pros and cons verdict wrap up"
}

event_centers = {}

for category, prompt in target_categories.items():
    prompt_vec = embedder.encode(prompt, convert_to_tensor=True)
    best_score = -1.0
    best_time = 0.0
    best_sentence = ""

    for seg in transcript:
        if not seg["text"]:
            continue
        seg_vec = embedder.encode(seg["text"], convert_to_tensor=True)
        score = util.cos_sim(prompt_vec, seg_vec).item()

        if score > best_score:
            best_score = score
            best_time = (seg["start"] + seg["end"]) / 2.0
            best_sentence = seg["text"]

    event_centers[category] = {
        "x": best_time,
        "confidence": round(best_score, 3),
        "matched_text": best_sentence
    }
    print(f"\n[{category}]")
    print(f"  Confidence : {best_score:.3f}")
    print(f"  Estimated x: {best_time:.2f}s")
    print(f"  Matched text: \"{best_sentence}\"")

Loading Whisper...


Extracted 257 timestamped speech segments.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


[Product Unboxing]
  Confidence : 0.310
  Estimated x: 294.76s
  Matched text: "that was like privacy, screen protectors,"

[Feature Demonstration]
  Confidence : 0.391
  Estimated x: 335.70s
  Matched text: "some of the brightness of the screen,"

[Final Verdict]
  Confidence : 0.240
  Estimated x: 404.54s
  Matched text: "When you compare them side by side,"


# **Buffer Window Calculation & Lossless Cut Demonstration**

In [5]:

y_buffer = 20.0


target_event = "Product Unboxing"
x_center = event_centers[target_event]["x"]


duration = min(max(2 * y_buffer, 30.0), 60.0)
start_time = max(0.0, x_center - (duration / 2.0))
end_time = start_time + duration

output_clip = f"week1_demo/{target_event.lower().replace(' ', '_')}_clip.mp4"


subprocess.run([
    "ffmpeg", "-y",
    "-ss", str(round(start_time, 2)),
    "-to", str(round(end_time, 2)),
    "-i", video_path,
    "-c", "copy",
    output_clip
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print(f"Successfully generated prototype clip for {target_event}:")
print(f"  Window: {start_time:.2f}s to {end_time:.2f}s (Total Duration: {duration:.2f}s)")
print(f"  Exported to: {output_clip}")

Successfully generated prototype clip for Product Unboxing:
  Window: 274.76s to 314.76s (Total Duration: 40.00s)
  Exported to: week1_demo/product_unboxing_clip.mp4
